In [3]:
#==========================================
# R.O.M. HOLOGRAPHIC DECODER: S0-2 EMPIRICAL DATA
# Pure Algebraic Exact Inversion with Deterministic Rømer Un-Warp
# ==========================================

import numpy as np
import pandas as pd
from scipy.optimize import differential_evolution, root_scalar
import warnings
warnings.filterwarnings('ignore')

# ---------------------------------------------------------
# CONSTANTS
# ---------------------------------------------------------
C_KMS = 299792.458

# ---------------------------------------------------------
# 0. LOAD EMPIRICAL DATA
# ---------------------------------------------------------
print("Loading empirical dataset (LSR-corrected)...")
df = pd.read_csv('https://raw.githubusercontent.com/AntonRize/WILL/refs/heads/main/DATA/S0-2_DataS1_full.csv')
df.columns = df.columns.str.strip()

t_obs = df['MJD'].values
vz_obs = df['RV_km_s'].values
sigma_vz = df['sigma_km_s'].values

Z_obs = 1.0 + (vz_obs / C_KMS)
sigma_Z = sigma_vz / C_KMS

# ---------------------------------------------------------
# 1. R.O.M. STRICT RELATIONAL GEOMETRY ENGINE
# ---------------------------------------------------------
def get_idealised_phase_o(t_array, t_peri, T_period, e):
    zeta_target = (2 * np.pi / T_period) * (t_array - t_peri)
    orbit_counts = np.floor(zeta_target / (2 * np.pi))
    z_tar = zeta_target % (2 * np.pi)

    e_X = (1 + e) / (1 - e)
    e_Y = np.sqrt(1 - e**2)

    o = z_tar.copy()

    for _ in range(25):
        term1 = 2 * np.arctan2(np.sin(o / 2), np.sqrt(e_X) * np.cos(o / 2))
        term1 = term1 % (2 * np.pi)
        term2 = (e * e_Y * np.sin(o)) / (1 + e * np.cos(o))

        zeta_current = term1 - term2
        error = zeta_current - z_tar

        if np.max(np.abs(error)) < 1e-12:
            break

        derivative = (e_Y**3) / ((1 + e * np.cos(o))**2)
        o = o - error / derivative

    return o + orbit_counts * 2 * np.pi

def generate_z_raw_will(o_unwrapped, beta, i_inc, beta_z0, e, omega_i):
    Omega_phi = (3 * beta**2 - 2 * beta**4) / (1 - e**2)
    Omega = 1.0 - Omega_phi

    O_unwrapped = Omega * o_unwrapped
    O = O_unwrapped % (2 * np.pi)
    e_Y = np.sqrt(1 - e**2)

    beta_int = beta / e_Y
    beta_z = beta_int * (np.cos(O + omega_i) + e * np.cos(omega_i)) * np.sin(i_inc)

    beta_o_sq = (beta**2) * (1 + e**2 + 2 * e * np.cos(O)) / (1 - e**2)
    kappa_o_sq = 2 * (beta**2) * (1 + e * np.cos(O)) / (1 - e**2)

    if np.any(beta_o_sq >= 1.0) or np.any(kappa_o_sq >= 1.0):
        return np.full_like(O, np.nan)

    Z_sys = (1 - beta_o_sq)**(-0.5) * (1 - kappa_o_sq)**(-0.5)

    return Z_sys * (1 + beta_z) * (1 + beta_z0)

# ---------------------------------------------------------
# 2. STAGE 1: MACROSCOPIC SCOUT (RIGID APPARENT TIMELINE)
# ---------------------------------------------------------
def scout_objective(params):
    beta, i_inc, vz0_c, e, omega, P, t_peri = params
    try:
        o_unwrapped = get_idealised_phase_o(t_obs, t_peri, P, e)
        Z_model = generate_z_raw_will(o_unwrapped, beta, i_inc, vz0_c, e, omega)
        if np.any(np.isnan(Z_model)): return 1e10
        return np.sum(((Z_obs - Z_model) / sigma_Z)**2)
    except:
        return 1e10

bounds_scout = [
    (0.003, 0.015),
    (0.0, np.pi/2),
    (-100.0/C_KMS, 0.0),
    (0.80, 0.95),
    (0.0, 2*np.pi),
    (15.5 * 365.25, 16.5 * 365.25),
    (58240.0, 58270.0)
]

print("\nSTAGE 1: Macroscopic Scout (Extracting Apparent Skeleton)...")
res_scout = differential_evolution(scout_objective, bounds_scout, strategy='best1bin', maxiter=1000, popsize=30, tol=1e-3, seed=42)

beta_s1, i_s1, vz0_c_s1, e_s1, omega_s1, P_s1, t_peri_apparent = res_scout.x
e_Y_s1 = np.sqrt(1 - e_s1**2)
K_macro_rel = (beta_s1 / e_Y_s1) * np.sin(i_s1)
vz0_s1 = vz0_c_s1 * C_KMS

print(f"-> Apparent Skeleton Locked: P = {P_s1/365.25:.4f} yrs, Apparent T_peri = {t_peri_apparent:.3f} MJD")

# ---------------------------------------------------------
# 3. STAGE 1.5: ALGEBRAIC RØMER UN-WARP
# ---------------------------------------------------------
print("\nSTAGE 1.5: Algebraic Rømer Un-Warping (Extracting True Intrinsic Timeline)...")

# Calculate exact intrinsic Time of Periapsis (O = 0)
delay_frac_peri = (K_macro_rel * (1 - e_s1**2)**1.5) / (2 * np.pi * (1 + e_s1)) * np.sin(omega_s1)
t_peri_true = t_peri_apparent - (delay_frac_peri * P_s1)

print(f"-> Intrinsic T_peri Extracted: {t_peri_true:.3f} MJD (Shift: {-(delay_frac_peri * P_s1):.3f} days)")

# Iterative solver to un-warp Earth observation times (t_obs) to True Emission times (t_emit)
Omega_phi_s1 = (3 * beta_s1**2 - 2 * beta_s1**4) / (1 - e_s1**2)
Omega_s1 = 1.0 - Omega_phi_s1

t_emit_array = t_obs.copy()
for _ in range(5):
    o_id_temp = get_idealised_phase_o(t_emit_array, t_peri_true, P_s1, e_s1)
    O_temp = (Omega_s1 * o_id_temp) % (2 * np.pi)

    # The pure WILL RG algebraic Rømer delay fraction
    delay_fraction = (K_macro_rel * (1 - e_s1**2)**1.5) / (2 * np.pi * (1 + e_s1 * np.cos(O_temp))) * np.sin(O_temp + omega_s1)
    t_emit_array = t_obs - (delay_fraction * P_s1)

# ---------------------------------------------------------
# 4. STAGE 2: HOLOGRAPHIC ALGEBRAIC DECRYPTION
# ---------------------------------------------------------
print("\nSTAGE 2: Algebraic Decryption (Solving Exact Kinetic Scale)...")

beta_min_limit = K_macro_rel * e_Y_s1

# Generate precise precessing phases (O) using the intrinsic emission timeline
o_unwrapped_emit = get_idealised_phase_o(t_emit_array, t_peri_true, P_s1, e_s1)
O_wrapped_emit = (Omega_s1 * o_unwrapped_emit) % (2 * np.pi)

phase_max_red = (2 * np.pi - omega_s1) % (2 * np.pi)
phase_max_blue = (np.pi - omega_s1) % (2 * np.pi)

def extract_empirical_extremum(target_phase, z_array, O_array, is_max=True, smooth_window=5):
    diffs = np.abs(O_array - target_phase)
    diffs = np.minimum(diffs, 2*np.pi - diffs)
    closest_idx = np.argsort(diffs)[:smooth_window]

    if is_max:
        return np.max(z_array[closest_idx])
    else:
        return np.min(z_array[closest_idx])

beta_alg = None
for window in [5, 7, 9]:
    Z_max_emp = extract_empirical_extremum(phase_max_red, Z_obs, O_wrapped_emit, is_max=True, smooth_window=window)
    Z_min_emp = extract_empirical_extremum(phase_max_blue, Z_obs, O_wrapped_emit, is_max=False, smooth_window=window)

    def decryption_invariant_residual(beta):
        if beta <= beta_min_limit or beta > 0.05: return 1e9

        K_sq1 = 2 * beta**2 * (1 + e_s1 * np.cos(omega_s1)) / (1 - e_s1**2)
        B_sq1 = beta**2 * (1 + e_s1**2 + 2 * e_s1 * np.cos(omega_s1)) / (1 - e_s1**2)
        tau1 = np.sqrt(1 - K_sq1) * np.sqrt(1 - B_sq1)

        K_sq2 = 2 * beta**2 * (1 - e_s1 * np.cos(omega_s1)) / (1 - e_s1**2)
        B_sq2 = beta**2 * (1 + e_s1**2 - 2 * e_s1 * np.cos(omega_s1)) / (1 - e_s1**2)
        tau2 = np.sqrt(1 - K_sq2) * np.sqrt(1 - B_sq2)

        inv = Z_max_emp * tau1 * (1 - e_s1 * np.cos(omega_s1)) + Z_min_emp * tau2 * (1 + e_s1 * np.cos(omega_s1))
        return inv - 2.0

    try:
        res_root = root_scalar(decryption_invariant_residual, bracket=[beta_min_limit + 1e-6, 0.03], method='brentq')
        beta_alg = res_root.root
        break
    except ValueError:
        continue

if beta_alg is None:
    raise ValueError("Empirical noise violates relational invariant capacity. Run halted.")

sin_i_alg = (K_macro_rel * e_Y_s1) / beta_alg
i_alg = np.arcsin(np.clip(sin_i_alg, 0.0, 1.0))
print(f"-> Algebraic Root Confirmed: beta = {beta_alg:.6f}, i = {np.degrees(i_alg):.2f} deg")

# ---------------------------------------------------------
# 5. PHYSICAL CALCULATIONS & FINAL OUTPUT
# ---------------------------------------------------------
delta_phi_will_alg = (2 * np.pi * (3 * beta_alg**2 - 2 * beta_alg**4)) / (1 - e_s1**2)
precession_deg_per_orbit = np.degrees(delta_phi_will_alg)

T_sec = P_s1 * 24 * 3600
Rs = T_sec * C_KMS * (beta_alg**3) / np.pi
a = Rs / (2 * (beta_alg**2))
m_rec = (Rs * 1000 * C_KMS**2 * 1e6) / (2 * 6.6743e-11) / 1.98847e30

print(f"\n{'='*65}")
print("=== R.O.M. PURE ALGEBRAIC RECOVERY (RØMER CORRECTED) ===")
print(f"{'='*65}")
print(f"Period (P):               {P_s1/365.25:.4f} yrs")
print(f"True Intrinsic T_p:       {t_peri_true:.3f} MJD")
print(f"Eccentricity (e):         {e_s1:.5f}")
print(f"Arg of Periapsis (w0):    {np.degrees(omega_s1) % 360:.2f} deg")
print(f"Internal Precession:      {precession_deg_per_orbit:.3f} deg/orbit")
print("-" * 65)
print(f"Global Kin. Proj (beta):  {beta_alg:.6f}")
print(f"Inclination (i_extracted):{np.degrees(i_alg):.2f} deg")
print(f"Background Drift (v_z0):  {vz0_s1:.2f} km/s")
print("-" * 65)
print(f"Schwarzschild Radius:     {Rs:.2f} km")
print(f"Semi-major axis (a):      {a:.2e} km")
print(f"Calculated Mass (M_sun):  {m_rec:,.2f}")
print(f"{'='*65}")

Loading empirical dataset (LSR-corrected)...

STAGE 1: Macroscopic Scout (Extracting Apparent Skeleton)...
-> Apparent Skeleton Locked: P = 16.0658 yrs, Apparent T_peri = 58259.694 MJD

STAGE 1.5: Algebraic Rømer Un-Warping (Extracting True Intrinsic Timeline)...
-> Intrinsic T_peri Extracted: 58259.195 MJD (Shift: -0.499 days)

STAGE 2: Algebraic Decryption (Solving Exact Kinetic Scale)...
-> Algebraic Root Confirmed: beta = 0.006260, i = 48.99 deg

=== R.O.M. PURE ALGEBRAIC RECOVERY (RØMER CORRECTED) ===
Period (P):               16.0658 yrs
True Intrinsic T_p:       58259.195 MJD
Eccentricity (e):         0.87694
Arg of Periapsis (w0):    66.81 deg
Internal Precession:      0.183 deg/orbit
-----------------------------------------------------------------
Global Kin. Proj (beta):  0.006260
Inclination (i_extracted):48.99 deg
Background Drift (v_z0):  -24.92 km/s
-----------------------------------------------------------------
Schwarzschild Radius:     11869332.40 km
Semi-major axis 

In [1]:
#!/usr/bin/env python3
# =============================================================================
# TRIANGULATION TEST  --  recover beta (hence M) from three balance points,
# bypassing the Z_max data gap.  WILL / R.O.M.  S2 / Sgr A*.
#
# THE IDEA (plain language)
# -------------------------
# A radial-velocity orbit alone fixes only the product M*sin(i): the Doppler
# amplitude gives K_i = (beta/e_Y) sin(i), so beta and inclination trade off.
# R.O.M. breaks that with the transverse+gravitational redshift Z_sys, which
# depends on beta and eccentricity ONLY (no inclination in it). Divide the
# Doppler out of each measured redshift and what is left, Z_sys(O), fixes beta.
#
# The two-point "invariant" reads beta off the redshift PEAK at O = -omega, which
# sits in the S2 sampling gap -> biased. TRIANGULATION instead reads beta at
# THREE reference phases that are NOT in the gap and demands one consistent beta:
#     perihelion   O = 0            Z_sys = 1/[sqrt(1-b^2 e_X) sqrt(1-2b^2/(1-e))]
#     balance B_a  O = arccos(-e)   Z_sys = 1/[sqrt(1-b^2)   sqrt(1-2b^2)]   (beta_o=beta)
#     balance B_d  O = 2pi-arccos(-e)  same clean form as B_a
# At B_a and B_d the local kinetic projection equals the global beta exactly, so
# Z_sys there is a pure function of beta. Forcing beta(r_p)=beta(B_a)=beta(B_d)
# over-determines beta and pins it.
#
# WHAT THIS TEST DOES
# -------------------
# Generates synthetic S2 radial velocities from the exact R.O.M. model at the
# REAL S2 observation times with realistic noise and a KNOWN true beta, then runs
# the triangulation to recover beta. It reports recovery in three data scenarios:
#   (1) full data
#   (2) the Z_max peak deleted            (the "gap" the invariant dies on)
#   (3) BOTH the peak AND perihelion deleted (only the two balance points left)
# The orbit (P, e, omega, T_peri, K_i, v_z0) is taken as known -- it is fixed by
# the huge Doppler signal / astrometry; beta is the quantity triangulation solves.
# =============================================================================

import numpy as np
import pandas as pd
from scipy.optimize import brentq, minimize_scalar
import warnings
warnings.filterwarnings('ignore')

C = 299792.458          # speed of light [km/s]
G = 6.67430e-11         # [m^3 kg^-1 s^-2]
MSUN = 1.98847e30       # [kg]
DAY = 86400.0
YEAR = 365.25
DATA_URL = ('https://raw.githubusercontent.com/AntonRize/WILL/'
            'refs/heads/main/DATA/S0-2_DataS1_full.csv')


# ------------------------------------------------------------ R.O.M. pieces
def phase_from_time(t, tp, P_days, e):
    """Idealised phase o(t) by inverting the R.O.M. mean-anomaly integral zeta."""
    z = (2 * np.pi / P_days) * (t - tp)
    k = np.floor(z / (2 * np.pi)); z = z % (2 * np.pi)
    eX = (1 + e) / (1 - e); eY = np.sqrt(1 - e ** 2); o = z.copy()
    for _ in range(40):
        A = (2 * np.arctan2(np.sin(o / 2), np.sqrt(eX) * np.cos(o / 2))) % (2 * np.pi)
        o = o - (A - (e * eY * np.sin(o)) / (1 + e * np.cos(o)) - z) / ((eY ** 3) / ((1 + e * np.cos(o)) ** 2))
    return o + k * 2 * np.pi


def Zsys(O, beta, e):
    """Transverse + gravitational redshift at true phase O. Depends on beta and
    e ONLY -- there is no inclination here. This is what carries beta."""
    bo = beta ** 2 * (1 + e ** 2 + 2 * e * np.cos(O)) / (1 - e ** 2)
    ko = 2 * beta ** 2 * (1 + e * np.cos(O)) / (1 - e ** 2)
    return (1 - bo) ** -0.5 * (1 - ko) ** -0.5


def rv_forward(O, beta, e, w, Ki, vz0):
    """Full observed radial velocity [km/s] at phase O (used to make synthetic data).
    Doppler carries only K_i; Z_sys carries only beta."""
    Doppler = Ki * (np.cos(O + w) + e * np.cos(w))
    return ((1 + Doppler) * Zsys(O, beta, e) * (1 + vz0) - 1) * C


# ------------------------------------------------------------ triangulation
def triangulation_beta(t, rv, sig, orbit, mask=None):
    """Recover beta by triangulation on perihelion + B_a + B_d.

    Steps:
      1. From the known orbit, get the true phase O(t) of every point and divide
         the Doppler out of the measured redshift to expose Z_sys_obs(O).
      2. Group points into three windows around the reference phases.
      3. In each window, solve for the single beta whose Z_sys(O,beta) best
         matches Z_sys_obs (error-weighted 1-D fit).
      4. Return the three per-point-set betas AND the combined beta that fits all
         three windows jointly (this is the 'beta(r_p)=beta(B_a)=beta(B_d)'
         consistency solve).
    """
    e, w, Ki, vz0, P_days, tp = orbit
    if mask is None:
        mask = np.ones(len(t), bool)
    sZ = sig / C
    Ba = np.arccos(-e); Bd = 2 * np.pi - np.arccos(-e)
    refs = {'perihelion': 0.0, 'B_a': Ba, 'B_d': Bd}
    win = np.radians(35.0)                              # +/-35 deg window per reference
    Zobs = 1.0 + rv / C

    def extract(beta_prec):
        # precession uses the current beta estimate -> self-consistent phase
        Om = 1.0 - (3 * beta_prec ** 2 - 2 * beta_prec ** 4) / (1 - e ** 2)
        O = (Om * phase_from_time(t, tp, P_days, e)) % (2 * np.pi)
        Doppler = Ki * (np.cos(O + w) + e * np.cos(w))
        Zsys_obs = (Zobs / (1 + vz0)) / (1 + Doppler)     # expose the transverse redshift

        def in_window(center):
            d = np.abs(O - center); d = np.minimum(d, 2 * np.pi - d)
            return mask & (d < win)

        def solve_beta(sel):
            if sel.sum() == 0:
                return np.nan, 0
            Osel, Zsel, ssel = O[sel], Zsys_obs[sel], sZ[sel]
            cost = lambda b: np.sum(((Zsel - Zsys(Osel, b, e)) / ssel) ** 2)
            grid = np.linspace(0.004, 0.011, 30)
            b0 = grid[np.argmin([cost(b) for b in grid])]
            r = minimize_scalar(cost, bounds=(max(0.004, b0 - 5e-4), min(0.011, b0 + 5e-4)),
                                method='bounded')
            return r.x, int(sel.sum())

        per = {name: solve_beta(in_window(c)) for name, c in refs.items()}
        sel_all = in_window(0.0) | in_window(Ba) | in_window(Bd)
        comb, n_all = solve_beta(sel_all)
        return per, comb, n_all

    # pass 1 with a nominal precession beta, then pass 2 self-consistent
    _, comb1, _ = extract(0.006)
    beta_prec = comb1 if np.isfinite(comb1) else 0.006
    per_point, beta_combined, n_all = extract(beta_prec)
    return per_point, beta_combined, n_all


def mass_from_beta(beta, e, P_days):
    Rs = P_days * DAY * C * beta ** 3 / np.pi
    a = Rs / (2 * beta ** 2)
    M = (Rs * 1e3) * (C * 1e3) ** 2 / (2 * G) / MSUN
    return M, Rs, a


# ------------------------------------------------------------ the test
def main(n_real=300):
    print("Loading real S2 observation times (for realistic sampling)...")
    df = pd.read_csv(DATA_URL); df.columns = df.columns.str.strip()
    t = df['MJD'].values
    sig = np.full(len(t), 25.0)          # representative RV error

    # ---- true (synthetic) S2 state ----
    e, w, i = 0.87694, np.radians(66.81), np.radians(48.99)
    beta_true = 0.006260; eY = np.sqrt(1 - e ** 2)
    Ki = (beta_true / eY) * np.sin(i)
    P_days, tp, vz0 = 16.0658 * YEAR, 58259.195, -24.92 / C
    orbit = (e, w, Ki, vz0, P_days, tp)

    Om = 1.0 - (3 * beta_true ** 2) / (1 - e ** 2)
    O = (Om * phase_from_time(t, tp, P_days, e)) % (2 * np.pi)
    rv_clean = rv_forward(O, beta_true, e, w, Ki, vz0)

    # sampling around the reference phases (deg to nearest data point)
    def gap_to(target_deg):
        d = np.abs(np.degrees(O) - target_deg); d = np.minimum(d, 360 - d)
        return d.min(), int((d < 15).sum())
    Ba_deg = np.degrees(np.arccos(-e)); Bd_deg = 360 - Ba_deg
    Zmax_deg = (360 - np.degrees(w)) % 360
    print(f"  N = {len(t)} points\n")
    print("Sampling near each phase (nearest point, # within 15 deg):")
    for nm, dg in [("perihelion 0", 0.0), ("B_a", Ba_deg), ("B_d", Bd_deg), ("Z_max peak", Zmax_deg)]:
        near, cnt = gap_to(dg)
        print(f"  {nm:14s} O={dg:6.1f} deg   nearest {near:5.1f} deg   ({cnt} pts within 15 deg)")

    # transverse signal size (km/s) at each reference
    print("\nTransverse signal that carries beta (km/s):")
    for nm, Oref in [("perihelion", 0.0), ("B_a", np.arccos(-e)), ("B_d", 2*np.pi-np.arccos(-e))]:
        print(f"  {nm:11s}: {(Zsys(Oref, beta_true, e) - 1) * C:6.1f} km/s")

    # define the three data scenarios via masks on phase
    dd = np.abs(np.degrees(O) - Zmax_deg); dd = np.minimum(dd, 360 - dd)
    dp = np.abs(np.degrees(O) - 0.0);      dp = np.minimum(dp, 360 - dp)
    full   = np.ones(len(t), bool)
    gap    = dd > 20.0                       # delete Z_max peak
    gap2   = (dd > 20.0) & (dp > 25.0)       # delete Z_max peak AND perihelion

    scenarios = [("FULL data", full),
                 ("Z_max PEAK deleted", gap),
                 ("Z_max peak AND perihelion deleted (balance points only)", gap2)]

    print("\n" + "=" * 74)
    print(f"TRIANGULATION RECOVERY  (true beta = {beta_true:.6f}, "
          f"true M = {mass_from_beta(beta_true, e, P_days)[0]:,.0f} Msun)")
    print("=" * 74)
    rng = np.random.default_rng(0)
    for label, mask in scenarios:
        # per-point-set betas on ONE clean realisation (to show consistency)
        per, comb, _ = triangulation_beta(t, rv_clean, sig, orbit, mask)
        # scatter over many noisy realisations for the combined estimate
        combs = []
        for _ in range(n_real):
            rv_n = rv_clean + rng.normal(0, sig)
            _, cb, _ = triangulation_beta(t, rv_n, sig, orbit, mask)
            combs.append(cb)
        combs = np.array(combs); combs = combs[np.isfinite(combs)]
        M, Rs, a = mass_from_beta(np.mean(combs), e, P_days)
        print(f"\n{label}  ({int(mask.sum())} of {len(t)} points):")
        line = "   per reference (clean): "
        for nm in ('perihelion', 'B_a', 'B_d'):
            b, n = per[nm]
            line += f"{nm}={b:.5f}(n={n})  " if np.isfinite(b) else f"{nm}=--  "
        print(line)
        print(f"   combined beta = {np.mean(combs):.6f} +/- {np.std(combs):.6f}  "
              f"(bias {100*(np.mean(combs)-beta_true)/beta_true:+.1f}%, scatter {100*np.std(combs)/beta_true:.1f}%)")
        print(f"   => M = {M:,.0f} Msun  (true 4,018,000)   R_s = {Rs:,.0f} km   sin i fixed => i recoverable")
    print("\n" + "=" * 74)
    print("READ-OFF: if the combined beta stays at the true value with small scatter")
    print("in ALL THREE scenarios -- including when the Z_max peak is gone and even")
    print("when only the two balance points remain -- then triangulation recovers the")
    print("hidden beta and breaks M*sin(i) without ever needing the missing peak.")
    print("=" * 74)


if __name__ == '__main__':
    main()

Loading real S2 observation times (for realistic sampling)...
  N = 82 points

Sampling near each phase (nearest point, # within 15 deg):
  perihelion 0   O=   0.0 deg   nearest   2.5 deg   (6 pts within 15 deg)
  B_a            O= 151.3 deg   nearest   0.3 deg   (12 pts within 15 deg)
  B_d            O= 208.7 deg   nearest   4.4 deg   (15 pts within 15 deg)
  Z_max peak     O= 293.2 deg   nearest   7.9 deg   (1 pts within 15 deg)

Transverse signal that carries beta (km/s):
  perihelion :  185.2 km/s
  B_a        :   17.6 km/s
  B_d        :   17.6 km/s

TRIANGULATION RECOVERY  (true beta = 0.006260, true M = 4,018,711 Msun)

FULL data  (82 of 82 points):
   per reference (clean): perihelion=0.00626(n=8)  B_a=0.00642(n=27)  B_d=0.00614(n=44)  
   combined beta = 0.006256 +/- 0.000169  (bias -0.1%, scatter 2.7%)
   => M = 4,010,402 Msun  (true 4,018,000)   R_s = 11,844,078 km   sin i fixed => i recoverable

Z_max PEAK deleted  (80 of 82 points):
   per reference (clean): perihelion=0.